### load packages

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from matplotlib import cm



##

# =========================
# VIRIDIS COLOR SCHEMA
# =========================
VIRIDIS = plt.colormaps["viridis"]

VIRIDIS_DARK   = VIRIDIS(0.12)
VIRIDIS_BLUE   = VIRIDIS(0.28)
VIRIDIS_TEAL   = VIRIDIS(0.45)
VIRIDIS_GREEN  = VIRIDIS(0.68)
VIRIDIS_YELLOW = VIRIDIS(0.92)

TITLE_GRAY = "#666666"
TICK_GRAY = "#7A7A7A"
AXIS_LINE = "#BFBFBF"

def rgba_from_mpl(c, alpha=None):
    r, g, b, a = c
    if alpha is None:
        alpha = a
    return f"rgba({int(r*255)}, {int(g*255)}, {int(b*255)}, {alpha:.3f})"

def build_plotly_viridis_colors(n, alpha=0.90):
    cmap = plt.colormaps["viridis"].resampled(n)
    return [rgba_from_mpl(cmap(i), alpha=alpha) for i in range(n)]

MARKET_COLOR_PLOTLY = rgba_from_mpl(VIRIDIS_DARK, 1.0)
CSAD_COLOR_PLOTLY   = rgba_from_mpl(VIRIDIS_GREEN, 1.0)
CSSD_COLOR_PLOTLY   = rgba_from_mpl(VIRIDIS_BLUE, 1.0)
EVENT_COLOR_PLOTLY  = rgba_from_mpl(VIRIDIS_YELLOW, 1.0)


### load data

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path("/Users/HUBerlin/Desktop/Herding/Herding-main 2")

# Load crypto prices
coin_df = pd.read_csv(BASE / "Crypto_Price_20240108.csv")

# Load CRIX, because your CRIX file uses semicolon separator
crix_df = pd.read_csv(BASE / "new_crix.csv", sep=";")

# Clean column names
coin_df.columns = coin_df.columns.str.strip()
crix_df.columns = crix_df.columns.str.strip()

# Rename Date to date
coin_df = coin_df.rename(columns={"Date": "date"})
crix_df = crix_df.rename(columns={"price": "CRIX"})

# Convert dates
coin_df["date"] = pd.to_datetime(coin_df["date"])
crix_df["date"] = pd.to_datetime(crix_df["date"])

# Drop clearly problematic / unwanted columns
coin_df = coin_df.drop(columns=["LTC", "BSV", "RDD", "BLK", "FTC"], errors="ignore")

# Convert coin price columns to numeric
for col in coin_df.columns:
    if col != "date":
        coin_df[col] = pd.to_numeric(coin_df[col], errors="coerce")

crix_df["CRIX"] = pd.to_numeric(crix_df["CRIX"], errors="coerce")

# Merge by date
price_df = coin_df.merge(crix_df, on="date", how="inner")

# Set date index
price_df = price_df.set_index("date").sort_index()

# Use the common useful period
price_df = price_df.loc["2020-01-01":"2024-04-07"]

print(price_df.shape)
print(price_df.index.min(), price_df.index.max())
display(price_df.head())
display(price_df.tail())


In [ ]:
from pathlib import Path
import pandas as pd

BASE = Path("/Users/HUBerlin/Desktop/Herding/Herding-main 2")

# load VCRIX file safely
vcrix_df = pd.read_csv(BASE / "vcrix.csv", sep=None, engine="python")

# clean column names
vcrix_df.columns = vcrix_df.columns.str.strip()

# check columns
print(vcrix_df.columns.tolist())
display(vcrix_df.head())

# make date column name consistent
vcrix_df = vcrix_df.rename(columns={"Date": "date"})

# convert date and set index
vcrix_df["date"] = pd.to_datetime(vcrix_df["date"])
vcrix_df = vcrix_df.set_index("date")

# filter from 2022 onward
vcrix_df = vcrix_df[vcrix_df.index >= "2022-01-01"]

vcrix_df.head()


In [ ]:
# Log returns
return_df = np.log(price_df).diff()

# Do NOT use dropna(how="any")
# Only remove rows where CRIX is missing
return_df = return_df.dropna(subset=["CRIX"])

print(return_df.shape)
print(return_df.index.min(), return_df.index.max())
display(return_df.head())

In [ ]:
fig = make_subplots(rows=1, cols=1, shared_xaxes=True, vertical_spacing=0)

fig.update_layout(
    height=600,
    width=1200,
    title='',
    showlegend=False,
    font=dict(family='Times New Roman', size=14),
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
)


fig.add_trace(go.Scatter(
    y=vcrix_df['vcrix'],
    x=vcrix_df.index,
    line=dict(color='black', width=1),
    # mode='markers', marker_size=1,
    ), row=1, col=1)


fig.update_xaxes(
    title='',
    tickformat='%Y-%m',
    # dtick='M3',
    showline=True,
    linewidth=1,
    linecolor='black',
    mirror=True,
    showgrid=False,
    row=1, col=1
)

fig.update_yaxes(
    showticklabels=True,
    title='VCRIX',
    showline=True,
    linewidth=1,
    linecolor='black',
    mirror=True,
    showgrid=False,
)

fig.show()


#### CSAD

In [ ]:
# Use cleaned log-return dataframe
df_returns = return_df.copy()
df_returns.index = pd.to_datetime(df_returns.index)
df_returns = df_returns.sort_index()

# Select 2022–2023
df_2022_2023 = df_returns.loc["2022-01-01":"2023-12-31"].copy()

print("df_2022_2023 shape:", df_2022_2023.shape)
print("Start:", df_2022_2023.index.min())
print("End:", df_2022_2023.index.max())

# Separate coin returns and CRIX market return
coin_returns = df_2022_2023.drop(columns=["CRIX"], errors="ignore")
market_returns = df_2022_2023["CRIX"]

# Keep only days with enough valid coins
valid_N = coin_returns.notna().sum(axis=1)

coin_returns = coin_returns[valid_N >= 5]
market_returns = market_returns.loc[coin_returns.index]
valid_N = coin_returns.notna().sum(axis=1)

print("coin_returns shape:", coin_returns.shape)
print("market_returns shape:", market_returns.shape)

In [ ]:
# CSAD: Cross-Sectional Absolute Deviation
csad_values = coin_returns.sub(market_returns, axis=0).abs().mean(axis=1)

csad_df = pd.DataFrame({
    "csad": csad_values,
    "CRIX": market_returns
})

csad_df["abs_CRIX"] = csad_df["CRIX"].abs()
csad_df["sq_CRIX"] = csad_df["CRIX"] ** 2

print("csad_df shape:", csad_df.shape)
display(csad_df.head())

In [ ]:
# CSSD: Cross-Sectional Standard Deviation
deviation_sq = coin_returns.sub(market_returns, axis=0) ** 2

cssd_values = np.sqrt(deviation_sq.sum(axis=1) / (valid_N - 1))

cssd_df = pd.DataFrame({
    "cssd": cssd_values,
    "CRIX": market_returns
})

cssd_df["abs_CRIX"] = cssd_df["CRIX"].abs()
cssd_df["sq_CRIX"] = cssd_df["CRIX"] ** 2

print("cssd_df shape:", cssd_df.shape)
display(cssd_df.head())

In [ ]:
## from ming bing
print("coin_returns shape:", coin_returns.shape)
print("coin_returns start:", coin_returns.index.min())
print("coin_returns end:", coin_returns.index.max())
print("coin_returns non-NA values:")
print(coin_returns.notna().sum())

print("\ncsad_df shape:", csad_df.shape)
print("csad_df start:", csad_df.index.min())
print("csad_df end:", csad_df.index.max())
print("csad non-NA:", csad_df["csad"].notna().sum())

display(coin_returns.head())
display(csad_df.head())

# =========================
# Viridis colors
# =========================
colors = build_plotly_viridis_colors(len(coin_returns.columns), alpha=0.65)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0)

fig.update_layout(
    height=800,
    width=1000,
    title='',
    showlegend=False,
    font=dict(family='Times New Roman', size=14, color=TICK_GRAY),
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
)

# Top panel: log returns
for i, col in enumerate(coin_returns.columns):
    fig.add_trace(
        go.Scatter(
            y=coin_returns[col],
            x=coin_returns.index,
            mode='lines',
            name=col,
            line=dict(width=1, color=colors[i % len(colors)]),
            opacity=0.55,
        ),
        row=1, col=1
    )

# Bottom panel: CSAD
fig.add_trace(
    go.Scatter(
        y=csad_df['csad'],
        x=csad_df.index,
        line=dict(color=CSAD_COLOR_PLOTLY, width=1.5),
    ),
    row=2, col=1
)

fig.update_xaxes(
    title='',
    showticklabels=False,
    showline=True,
    linewidth=1,
    linecolor=AXIS_LINE,
    mirror=True,
    showgrid=False,
    tickfont=dict(color=TICK_GRAY),
    row=1, col=1
)

fig.update_xaxes(
    title='',
    tickformat='%Y-%m',
    dtick='M1',
    showline=True,
    linewidth=1,
    linecolor=AXIS_LINE,
    mirror=True,
    showgrid=False,
    tickfont=dict(color=TICK_GRAY),
    row=2, col=1
)

fig.update_yaxes(
    showticklabels=True,
    title='Log return',
    showline=True,
    linewidth=1,
    linecolor=AXIS_LINE,
    mirror=True,
    showgrid=False,
    tickfont=dict(color=TICK_GRAY),
    title_font=dict(color=TICK_GRAY),
    row=1, col=1,
)

fig.update_yaxes(
    showticklabels=True,
    title='CSAD',
    showline=True,
    linewidth=1,
    linecolor=AXIS_LINE,
    mirror=True,
    showgrid=False,
    tickfont=dict(color=TICK_GRAY),
    title_font=dict(color=TICK_GRAY),
    row=2, col=1
)

fig.show()


In [ ]:
import sys

!{sys.executable} -m pip uninstall -y kaleido
!{sys.executable} -m pip install kaleido==0.2.1

In [ ]:
import sys
#!{sys.executable} -m pip install -U kaleido

from pathlib import Path

BASE = Path("/Users/HUBerlin/Desktop/Herding/Herding-main 2")

fig.write_image(
    BASE / "csad_logreturn_plot.png",
    width=1200,
    height=800,
    scale=3
)

print(BASE / "csad_logreturn_plot.png")
print((BASE / "csad_logreturn_plot.png").exists())

fig.write_image(
    "csad_logreturn_plot.png",
    width=1200,
    height=800,
    scale=3
)

In [ ]:

#anime_df = pd.DataFrame()
#anime_df['csad'] = csad_df['csad']
#anime_df['CRIX'] = CRIX[(CRIX.index >= '2022-01-01') & (CRIX.index <= '2023-12-31')]

anime_df = pd.DataFrame()

anime_df["csad"] = csad_df["csad"]
anime_df["CRIX"] = csad_df["CRIX"]

anime_df = anime_df.loc["2022-01-01":"2023-12-31"]

display(anime_df.head())
print(anime_df.shape)


### static plot between csad and crix

In [ ]:
anime_df = pd.DataFrame()

anime_df["csad"] = csad_df["csad"]
anime_df["CRIX"] = csad_df["CRIX"]

# create duplicate name so old code using CRIX also works
anime_df["CRIX"] = anime_df["CRIX"]

anime_df = anime_df.loc["2022-01-01":"2023-12-31"].dropna().copy()
anime_df.index = pd.to_datetime(anime_df.index)
anime_df = anime_df.sort_index()

print(anime_df.columns)

fig = make_subplots(rows=1, cols=1, shared_xaxes=True, vertical_spacing=0)

fig.update_layout(
    height=600,
    width=600,
    title='',
    showlegend=False,  
    font=dict(family='Times New Roman', size=14),
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
)
    
fig.add_trace(go.Scatter(
    y=anime_df['csad'], 
    x=anime_df['CRIX'],
    line=dict(color='black', width=1),
    mode='markers', marker_size=10, 
    ), row=1, col=1)



# Update x-axis for the first row
fig.update_xaxes(
    title='Expected market return',
    showticklabels=True,  # Hide tick labels
    showline=True,
    linewidth=1,
    linecolor='black',
    mirror=True,
    showgrid=False,
    row=1, col=1,
    range=[-0.2,0.2]
)


# Update y-axis for the first row
fig.update_yaxes(
    showticklabels=True,
    title='CSAD',
    showline=True,
    linewidth=1,
    linecolor='black',
    mirror=True,
    showgrid=False,
    row=1, col=1,
    range=[0, 0.15]
)

# Show the plot
fig.show()


### anime without the marker of event dates

In [ ]:

anime_df = pd.DataFrame()
anime_df['csad'] = csad_df['csad']
anime_df = pd.DataFrame()

anime_df["csad"] = csad_df["csad"]
anime_df["CRIX"] = csad_df["CRIX"]

anime_df = anime_df.loc["2022-01-01":"2023-12-31"]

display(anime_df.head())
print(anime_df.shape)

event_labels = {
    pd.Timestamp("2022-05-13"): "Terra's Crash",
    pd.Timestamp("2022-11-11"): "FTX Bankruptcy",
    pd.Timestamp("2023-07-13"): "Ripple Wins a Partial Victory Against SEC",
}

event_rows = []

for event_date, label in event_labels.items():
    if event_date in anime_df.index:
        nearest_date = event_date
    else:
        nearest_date = anime_df.index[np.argmin(np.abs(anime_df.index - event_date))]

    event_rows.append({
        "date": nearest_date,
        "label": label,
        "x": anime_df.loc[nearest_date, "CRIX"],
        "y": anime_df.loc[nearest_date, "csad"],
    })

event_df = pd.DataFrame(event_rows).drop_duplicates("date")
display(event_df)

fig = make_subplots(rows=1, cols=1)

fig.update_layout(
    height=600,
    width=600,
    showlegend=False,
    font=dict(family='Times New Roman', size=14),
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
)



fig.update_xaxes(
    showgrid=False,
    zeroline=False,
    showline=True,
    linecolor="lightgray",
    tickfont=dict(color="gray"),
    title_font=dict(color="gray")
)

fig.update_yaxes(
    showgrid=False,
    zeroline=False,
    showline=True,
    linecolor="lightgray",
    tickfont=dict(color="gray"),
    title_font=dict(color="gray")
)

# Generate a colormap from dark to light using 'viridis'
# Viridis time-gradient colors
# Generate a colormap from dark to light using 'viridis'
# Viridis time-gradient colors
# Viridis time-gradient colors
colors = build_plotly_viridis_colors(len(anime_df), alpha=0.90)

# Initial Trace
fig.add_trace(
    go.Scatter(
        x=anime_df['CRIX'],
        y=anime_df['csad'],
        mode='markers',
        marker=dict(size=10, color=colors[0])
    )
)

# Configure x and y axis
fig.update_xaxes(title='Expected market return', range=[-0.2, 0.2])
fig.update_yaxes(title='CSAD', range=[0, 0.15])



highlight_dates = set(event_df["date"])
# Create frames for animation
frames = [
    go.Frame(
        data=[
            go.Scatter(
                x=anime_df.iloc[:i+1]['CRIX'],
                y=anime_df.iloc[:i+1]['csad'],
                mode='markers',
                marker=dict(
                    size=[20 if anime_df.index[j] in highlight_dates else 10 for j in range(i + 1)],
    color=[
        colors[j] if anime_df.index[j] not in highlight_dates else EVENT_COLOR_PLOTLY
        for j in range(i + 1)
    ],
    symbol=['circle' if anime_df.index[j] not in highlight_dates else 'star' for j in range(i + 1)]
                )
            )
        ],
        name=f"frame_{i}"
    )
    for i in range(len(anime_df))
]

fig.frames = frames

fig.update_layout(
    updatemenus=[{
        "buttons": [
            {"args": [None, {"frame": {"duration": 20, "redraw": True}, "fromcurrent": True}], "label": "Play", "method": "animate"},
            {"args": [[None], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate", "transition": {"duration": 0}}], "label": "Pause", "method": "animate"}
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "type": "buttons",
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 20},
            "prefix": "Time:",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 20, "easing": "cubic-in-out"},
        "pad": {"b": 10, "t": 50},
        "len": 0.9,
        "x": 0.1,
        "y": 0,
        "steps": [{
            "args": [[f'frame_{i}'], {"frame": {"duration": 20, "redraw": True}, "mode": "immediate", "transition": {"duration": 10}}],
            "label": anime_df.index[i].strftime('%Y-%m-%d'),
            "method": "animate"
        } for i in range(len(anime_df))]
    }]
)

fig.show()

fig.write_html("animated_plot.html")

### anime with markers of the event dates

In [ ]:
# ============================================================
# Build animation dataframe
# ============================================================

# Make sure index is datetime
csad_df.index = pd.to_datetime(csad_df.index)

anime_df = pd.DataFrame(index=csad_df.index)

anime_df["csad"] = csad_df["csad"]
anime_df["CRIX"] = csad_df["CRIX"]

# Keep only 2022-2023 and remove missing values
anime_df = anime_df.loc["2022-01-01":"2023-12-31"].dropna().copy()
anime_df.index = pd.to_datetime(anime_df.index)
anime_df = anime_df.sort_index()

print(anime_df.columns)
print(anime_df.head())


# ============================================================
# Event dates for stars
# ============================================================

event_labels = {
    pd.Timestamp("2022-05-13"): "Terra's Crash",
    pd.Timestamp("2022-11-11"): "FTX Bankruptcy",
    pd.Timestamp("2023-07-13"): "Ripple's Win",
}

event_rows = []

for event_date, label in event_labels.items():
    if event_date in anime_df.index:
        nearest_date = event_date
    else:
        nearest_date = anime_df.index[np.argmin(np.abs(anime_df.index - event_date))]

    event_rows.append({
        "date": nearest_date,
        "label": label,
        "x": anime_df.loc[nearest_date, "CRIX"],
        "y": anime_df.loc[nearest_date, "csad"],
    })

event_df = pd.DataFrame(event_rows).drop_duplicates("date")

display(event_df)

fig = make_subplots(rows=1, cols=1)

fig.update_layout(
    height=600,
    width=600,
    showlegend=False,
    font=dict(family='Times New Roman', size=14),
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
)

# Generate a colormap from dark to light using 'viridis'
cmap = cm.get_cmap('viridis', len(anime_df))
colors = [cmap(i) for i in range(len(anime_df))]
colors = ['rgba({}, {}, {}, {})'.format(int(r*255), int(g*255), int(b*255), a) for r, g, b, a in colors]

# Initial Trace
fig.add_trace(
    go.Scatter(

        x=[anime_df["CRIX"].iloc[0]],
        y=[anime_df["csad"].iloc[0]],
        mode='markers',
        marker=dict(size=10, color=colors[0])
    )
)

# Event stars start empty and appear later with time
fig.add_trace(
    go.Scatter(
        x=[],
        y=[],
        mode="markers+text",
        text=[],
        textposition="top center",
        marker=dict(
            symbol="star",
            size=22,
            color=EVENT_COLOR_PLOTLY,
           line=dict(color=EVENT_COLOR_PLOTLY, width=1),
        ),
        textfont=dict(
            family="Times New Roman",
            size=13,
            color="rgb(50,70,110)"
        ),
        hoverinfo="skip"
    )
)

highlight_dates = set(event_df["date"])
# Configure x and y axis
fig.update_xaxes(title='Expected market return', range=[-0.2, 0.2])
fig.update_yaxes(title='CSAD', range=[0, 0.15])

# Create frames for animation
frames = [
    go.Frame(
        data=[
            go.Scatter(
                x=anime_df.iloc[:i+1]['CRIX'],
                y=anime_df.iloc[:i+1]['csad'],
                mode='markers',
                marker=dict(
                    size=[20 if anime_df.index[j] in highlight_dates else 10 for j in range(i+1)],
                    color=[colors[j] if anime_df.index[j] not in highlight_dates else EVENT_COLOR_PLOTLY
    for j in range(i + 1)],
                    symbol=['star' if anime_df.index[j] in highlight_dates else 'circle' for j in range(i+1)]
                )
            )
        ],
        name=f"frame_{i}",
        layout=go.Layout(
            annotations=[
                dict(
                    x=anime_df.loc[date]['CRIX'],
                    y=anime_df.loc[date]['csad'],
                    text=label,
                    showarrow=True,
                    arrowhead=2,
                    ax=20,
                    ay=-30
                ) for date, label in event_labels.items() if date in anime_df.index[:i+1]
            ]
        )
    )
    for i in range(len(anime_df))
]

fig.frames = frames

fig.update_layout(
    updatemenus=[{
        "buttons": [
            {"args": [None, {"frame": {"duration": 20, "redraw": True}, "fromcurrent": True}], "label": "Play", "method": "animate"},
            {"args": [[None], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate", "transition": {"duration": 0}}], "label": "Pause", "method": "animate"}
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "type": "buttons",
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 20},
            "prefix": "Time:",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 20, "easing": "cubic-in-out"},
        "pad": {"b": 10, "t": 50},
        "len": 0.9,
        "x": 0.1,
        "y": 0,
        "steps": [{
            "args": [[f'frame_{i}'], {"frame": {"duration": 20, "redraw": True}, "mode": "immediate", "transition": {"duration": 10}}],
            "label": anime_df.index[i].strftime('%Y-%m-%d'),
            "method": "animate"
        } for i in range(len(anime_df))]
    }]
)

fig.show()

fig.write_html("animated_plot.html")


In [ ]:
from pathlib import Path
from PIL import Image
import shutil
import warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from matplotlib import cm

warnings.filterwarnings("ignore", category=DeprecationWarning)

# ============================================================
# OUTPUT PATH
# ============================================================
BASE = Path("/Users/HUBerlin/Desktop/Herding/Herding-main 2")

FRAME_DIR = BASE / "csad_scatter_gif_frames_UPDATED"
GIF_PATH = BASE / "csad_scatter_transparent_UPDATED.gif"

# Delete old updated frames
if FRAME_DIR.exists():
    shutil.rmtree(FRAME_DIR)

FRAME_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# BUILD DATAFRAME — based on previous student's structure
# ============================================================
csad_df.index = pd.to_datetime(csad_df.index)

anime_df = pd.DataFrame(index=csad_df.index)
anime_df["csad"] = csad_df["csad"]

# Keep previous student's name: avg_CRIX
anime_df["avg_CRIX"] = csad_df["CRIX"]

anime_df = anime_df.loc["2022-01-01":"2023-12-31"].dropna().copy()
anime_df = anime_df.sort_index()

print("anime_df columns:", anime_df.columns.tolist())
print("Rows:", len(anime_df))
print("Start:", anime_df.index.min())
print("End:", anime_df.index.max())

# ============================================================
# EVENT DATES — stars appear only when time reaches them
# ============================================================
highlight_labels = {
    "2022-05-13": "Terra's Crash",
    "2022-11-11": "FTX Bankruptcy",
    "2023-07-13": "Ripple's Win",
}

event_rows = []

for raw_date, label in highlight_labels.items():
    event_date = pd.Timestamp(raw_date)

    if event_date in anime_df.index:
        nearest_date = event_date
    else:
        nearest_date = anime_df.index[np.argmin(np.abs(anime_df.index - event_date))]

    event_rows.append({
        "date": nearest_date,
        "label": label,
        "x": anime_df.loc[nearest_date, "avg_CRIX"],
        "y": anime_df.loc[nearest_date, "csad"],
    })

event_df = pd.DataFrame(event_rows).drop_duplicates("date")
display(event_df)

# ============================================================
# COLORS — viridis time-gradient
# ============================================================
colors = build_plotly_viridis_colors(len(anime_df), alpha=0.90)

# ============================================================
# ANIMATION SETTINGS
# ============================================================
FPS = 10
TARGET_DURATION_SEC = 20

target_frames = FPS * TARGET_DURATION_SEC
frame_step = max(1, int(np.ceil(len(anime_df) / target_frames)))

frames_to_save = list(range(0, len(anime_df), frame_step))
if frames_to_save[-1] != len(anime_df) - 1:
    frames_to_save.append(len(anime_df) - 1)

print("Frame step:", frame_step)
print("Frames:", len(frames_to_save))
print("Approx duration:", len(frames_to_save) / FPS, "seconds")

# ============================================================
# AXIS LIMITS
# ============================================================
x_min, x_max = anime_df["avg_CRIX"].min(), anime_df["avg_CRIX"].max()
y_min, y_max = anime_df["csad"].min(), anime_df["csad"].max()

x_pad = max((x_max - x_min) * 0.08, 1e-6)
y_pad = max((y_max - y_min) * 0.10, 1e-6)

# ============================================================
# SAVE PNG FRAMES
# ============================================================
for out_i, i in enumerate(frames_to_save):
    current = anime_df.iloc[: i + 1]
    current_date = anime_df.index[i]

    # events that should already be visible
    shown_events = event_df[event_df["date"] <= current_date]

    fig = go.Figure()

    # Main time-gradient scatter
    fig.add_trace(
        go.Scatter(
            x=current["avg_CRIX"],
            y=current["csad"],
            mode="markers",
            marker=dict(
                size=10,
                color=colors[: i + 1],
                opacity=0.90,
            ),
            hoverinfo="skip",
        )
    )

    # Event stars appear gradually
    if not shown_events.empty:
        fig.add_trace(
            go.Scatter(
                x=shown_events["x"],
                y=shown_events["y"],
                mode="markers+text",
                text=shown_events["label"],
                textposition="top center",
                marker=dict(
                    symbol="star",
                    size=24,
                    color="red",
                    line=dict(color=EVENT_COLOR_PLOTLY, width=1),
                ),
                textfont=dict(
                    family="Times New Roman",
                    size=14,
                    color="rgb(50,70,110)",
                ),
                hoverinfo="skip",
            )
        )

    fig.update_layout(
        height=650,
        width=850,
        showlegend=False,
        font=dict(family="Times New Roman", size=18, color="rgb(50,70,110)"),
        plot_bgcolor="rgba(0,0,0,0)",
        paper_bgcolor="rgba(0,0,0,0)",
        margin=dict(l=90, r=70, t=40, b=120),
        annotations=[
            dict(
                text=f"Time:{current_date.strftime('%Y-%m-%d')}",
                x=0.95,
                y=-0.16,
                xref="paper",
                yref="paper",
                showarrow=False,
                font=dict(
                    family="Times New Roman",
                    size=26,
                    color="rgb(50,70,110)",
                ),
            )
        ],
    )

    fig.update_xaxes(
        title="Expected market return",
        range=[x_min - x_pad, x_max + x_pad],
        showline=True,
        linewidth=1,
        linecolor="rgb(170,170,170)",
        mirror=True,
        showgrid=False,
        zeroline=False,
        tickfont=dict(color="rgb(50,70,110)", size=15),
        title_font=dict(color="rgb(50,70,110)", size=21),
    )

    fig.update_yaxes(
        title="CSAD",
        range=[0, y_max + y_pad],
        showline=True,
        linewidth=1,
        linecolor="rgb(170,170,170)",
        mirror=True,
        showgrid=False,
        zeroline=False,
        tickfont=dict(color="rgb(50,70,110)", size=15),
        title_font=dict(color="rgb(50,70,110)", size=21),
    )

    fig.write_image(
        str(FRAME_DIR / f"frame_{out_i:04d}.png"),
        scale=2,
    )

print("Saved frames to:", FRAME_DIR)

# ============================================================
# CONVERT PNG FRAMES TO TRANSPARENT GIF
# ============================================================
frame_files = sorted(FRAME_DIR.glob("frame_*.png"))

if not frame_files:
    raise RuntimeError("No frames were created. GIF cannot be generated.")

duration_ms = int(1000 / FPS)

gif_frames = []

for f in frame_files:
    img = Image.open(f).convert("RGBA")

    alpha = img.getchannel("A")
    paletted = img.convert("P", palette=Image.ADAPTIVE, colors=255)

    transparency_mask = Image.eval(alpha, lambda a: 255 if a <= 5 else 0)
    paletted.paste(255, mask=transparency_mask)
    paletted.info["transparency"] = 255

    gif_frames.append(paletted)

gif_frames[0].save(
    GIF_PATH,
    save_all=True,
    append_images=gif_frames[1:],
    duration=duration_ms,
    loop=0,
    transparency=255,
    disposal=2,
    optimize=False,
)

print("Saved updated transparent GIF:")
print(GIF_PATH)
print("Exists:", GIF_PATH.exists())
print("Size MB:", round(GIF_PATH.stat().st_size / 1024 / 1024, 2))

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt

# ============================================================
# Build anime_df safely
# ============================================================
anime_df = pd.DataFrame()

anime_df["csad"] = csad_df["csad"]
anime_df["CRIX"] = csad_df["CRIX"]

# Create CRIX too, so both old and new code work
anime_df["CRIX"] = anime_df["CRIX"]

anime_df = anime_df.loc["2022-01-01":"2023-12-31"].dropna().copy()
anime_df.index = pd.to_datetime(anime_df.index)
anime_df = anime_df.sort_index()

print(anime_df.columns)

anime_df = anime_df.loc["2022-01-01":"2023-12-31"].dropna().copy()
anime_df.index = pd.to_datetime(anime_df.index)
anime_df = anime_df.sort_index()

print(anime_df.columns)
print("Rows:", len(anime_df))

# ============================================================
# Event dates
# ============================================================
event_labels = {
    pd.Timestamp("2022-05-13"): "Terra's Crash",
    pd.Timestamp("2022-11-11"): "FTX Bankruptcy",
    pd.Timestamp("2023-07-13"): "Ripple Wins a Partial Victory Against SEC",
}

event_rows = []

for event_date, label in event_labels.items():
    if event_date in anime_df.index:
        nearest_date = event_date
    else:
        nearest_date = anime_df.index[np.argmin(np.abs(anime_df.index - event_date))]

    event_rows.append({
        "date": nearest_date,
        "label": label,
        "x": anime_df.loc[nearest_date, "CRIX"],
        "y": anime_df.loc[nearest_date, "csad"],
    })

event_df = pd.DataFrame(event_rows).drop_duplicates("date")

# ============================================================
# Time-gradient colors
# ============================================================
colors = build_plotly_viridis_colors(len(anime_df), alpha=0.90)

# ============================================================
# Build figure
# ============================================================


# ============================================================
# FIGURE SETUP + INITIAL TRACES + AXIS STYLE
# Paste this after colors are created and before frames = []
# ============================================================

fig = make_subplots(rows=1, cols=1)

fig.update_layout(
    height=1000,
    width=1200,
    showlegend=False,
    font=dict(
        family="Times New Roman",
        size=18,
        color="#2f4368"
    ),
    plot_bgcolor="rgba(0,0,0,0)",
    paper_bgcolor="rgba(0,0,0,0)",
    margin=dict(l=90, r=90, t=40, b=120),
)

# Initial CSAD scatter point
fig.add_trace(
    go.Scatter(
        x=[anime_df["CRIX"].iloc[0]],
        y=[anime_df["csad"].iloc[0]],
        mode="markers",
        marker=dict(
            size=14,
            color=[colors[0]],
            line=dict(width=0.8, color="white")
        ),
        hoverinfo="skip"
    ),
    row=1,
    col=1
)

# Event stars start empty.
# They will appear later inside the frame loop.
fig.add_trace(
    go.Scatter(
        x=[],
        y=[],
        mode="markers+text",
        text=[],
        textposition="top center",
        textfont=dict(
            family="Times New Roman",
            size=14,
            color="#2f4368"
        ),
        marker=dict(
            symbol="star",
            size=24,
            color="#e4574f",
            line=dict(color="#e4574f", width=1)
        ),
        hoverinfo="skip"
    ),
    row=1,
    col=1
)

# Initial time label
fig.add_annotation(
    x=0.82,
    y=-0.14,
    xref="paper",
    yref="paper",
    text=f"Time:{anime_df.index[0].strftime('%Y-%m-%d')}",
    showarrow=False,
    font=dict(
        family="Times New Roman",
        size=28,
        color="#2f4368"
    )
)

# X-axis style
fig.update_xaxes(
    title="Expected market return",
    showline=True,
    linewidth=2,
    linecolor="#2f4368",
    mirror=False,
    showgrid=False,
    zeroline=False,
    range=[-0.2, 0.22],
    tickfont=dict(
        family="Times New Roman",
        size=16,
        color="#2f4368"
    ),
    title_font=dict(
        family="Times New Roman",
        size=24,
        color="#2f4368"
    ),
    row=1,
    col=1
)

# Y-axis style
fig.update_yaxes(
    title="CSAD",
    showline=True,
    linewidth=2,
    linecolor="#2f4368",
    mirror=False,
    showgrid=False,
    zeroline=False,
    range=[0, 0.15],
    tickfont=dict(
        family="Times New Roman",
        size=16,
        color="#2f4368"
    ),
    title_font=dict(
        family="Times New Roman",
        size=24,
        color="#2f4368"
    ),
    row=1,
    col=1
)

# ============================================================
# Frames
# ============================================================
frames = []

for k in range(1, len(anime_df) + 1):
    current_date = anime_df.index[k - 1]
    current_date_str = current_date.strftime("%Y-%m-%d")

    # Only show event stars after their event date has arrived
    shown_events = event_df[event_df["date"] <= current_date]

    frames.append(
        go.Frame(
            data=[
                # Trace 0: animated CSAD scatter points
                go.Scatter(
                    x=anime_df["CRIX"].iloc[:k],
                    y=anime_df["csad"].iloc[:k],
                    mode="markers",
                    marker=dict(
                        size=10,
                        color=colors[:k],
                    ),
                    hovertemplate=
                        "Expected market return: %{x:.4f}<br>" +
                        "CSAD: %{y:.4f}<extra></extra>"
                ),

                # Trace 1: event stars appear only after event date
                go.Scatter(
                    x=shown_events["x"],
                    y=shown_events["y"],
                    mode="markers+text",
                    text=shown_events["label"],
                    textposition="top center",
                    marker=dict(
                        symbol="star",
                        size=22,
                        color=EVENT_COLOR_PLOTLY,
                       line=dict(color=EVENT_COLOR_PLOTLY, width=1),
                    ),
                    textfont=dict(
                        family="Times New Roman",
                        size=13,
                        color="rgb(50,70,110)"
                    ),
                    hoverinfo="skip"
                )
            ],
            traces=[0, 1],
            name=str(k),
            layout=go.Layout(
                annotations=[
                    dict(
                        text=f"Time:{current_date_str}",
                        x=0.95,
                        y=-0.15,
                        xref="paper",
                        yref="paper",
                        showarrow=False,
                        font=dict(
                            family="Times New Roman",
                            size=24,
                            color="rgb(50,70,110)"
                        )
                    )
                ]
            )
        )
    )

fig.frames = frames

# ============================================================
# Layout
# ============================================================
fig.update_layout(
    height=600,
    width=800,
    showlegend=False,
    font=dict(family="Times New Roman", size=16, color="rgb(50,70,110)"),
    plot_bgcolor="rgba(0,0,0,0)",
    paper_bgcolor="rgba(0,0,0,0)",
    margin=dict(l=80, r=80, t=40, b=120),
    annotations=[
        dict(
            text=f"Time:{anime_df.index[0].strftime('%Y-%m-%d')}",
            x=0.95,
            y=-0.15,
            xref="paper",
            yref="paper",
            showarrow=False,
            font=dict(
                family="Times New Roman",
                size=24,
                color="rgb(50,70,110)"
            )
        )
    ],
    updatemenus=[
        dict(
            type="buttons",
            showactive=False,
            x=0.02,
            y=-0.12,
            xanchor="left",
            yanchor="top",
            buttons=[
                dict(
                    label="Play",
                    method="animate",
                    args=[
                        None,
                        dict(
                            frame=dict(duration=80, redraw=True),
                            transition=dict(duration=0),
                            fromcurrent=True,
                            mode="immediate"
                        )
                    ]
                ),
                dict(
                    label="Pause",
                    method="animate",
                    args=[
                        [None],
                        dict(
                            frame=dict(duration=0, redraw=False),
                            transition=dict(duration=0),
                            mode="immediate"
                        )
                    ]
                )
            ]
        )
    ]
)

fig.update_xaxes(
    title="Expected market return",
    showline=True,
    linewidth=1,
    linecolor="rgb(160,170,190)",
    mirror=True,
    showgrid=False,
    zeroline=False,
    tickfont=dict(color="rgb(50,70,110)", size=14),
    title_font=dict(color="rgb(50,70,110)", size=20),
    range=[
        anime_df["CRIX"].min() - 0.02,
        anime_df["CRIX"].max() + 0.02
    ]
)

fig.update_yaxes(
    title="CSAD",
    showline=True,
    linewidth=1,
    linecolor="rgb(160,170,190)",
    mirror=True,
    showgrid=False,
    zeroline=False,
    tickfont=dict(color="rgb(50,70,110)", size=14),
    title_font=dict(color="rgb(50,70,110)", size=20),
    range=[
        0,
        anime_df["csad"].max() * 1.15
    ]
)

fig.show()

In [ ]:
from pathlib import Path
from PIL import Image
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import shutil
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)

BASE = Path("/Users/HUBerlin/Desktop/Herding/Herding-main 2")

FRAME_DIR = BASE / "csad_scatter_gif_frames_UPDATED"
GIF_PATH = BASE / "csad_scatter_transparent_UPDATED.gif"

FPS = 10
TARGET_DURATION_SEC = 20

# Clean and sort data
anime = anime_df.copy()
anime.index = pd.to_datetime(anime.index)
anime = anime.sort_index()

# Make sure these columns exist
print(anime.columns)

x_col = "CRIX"
y_col = "csad"

# Auto frame step for about 20 seconds
target_frames = FPS * TARGET_DURATION_SEC
frame_step = max(1, int(np.ceil(len(anime) / target_frames)))

frames_to_save = list(range(0, len(anime), frame_step))
if frames_to_save[-1] != len(anime) - 1:
    frames_to_save.append(len(anime) - 1)

print("Total data rows:", len(anime))
print("Frames to save:", len(frames_to_save))
print("Approx duration:", len(frames_to_save) / FPS, "seconds")

# Axis limits
x_min, x_max = anime[x_col].min(), anime[x_col].max()
y_min, y_max = anime[y_col].min(), anime[y_col].max()

x_pad = max((x_max - x_min) * 0.08, 1e-6)
y_pad = max((y_max - y_min) * 0.10, 1e-6)

# Prepare frame folder
if FRAME_DIR.exists():
    shutil.rmtree(FRAME_DIR)

FRAME_DIR.mkdir(parents=True, exist_ok=True)

# Save PNG frames
for out_i, frame in enumerate(frames_to_save):
    sub = anime.iloc[:frame + 1]
    current_date = anime.index[frame].strftime("%Y-%m-%d")

    fig_frame = make_subplots(rows=1, cols=1)

    fig_frame.add_trace(
        go.Scatter(
            x=sub[x_col],
            y=sub[y_col],
            mode="markers",
            marker=dict(
    size=10,
    color=build_plotly_viridis_colors(len(sub), alpha=0.90),
    opacity=0.90
)
        )
    )

    fig_frame.update_layout(
        height=700,
        width=900,
        showlegend=False,
        font=dict(family="Times New Roman", size=20, color="#7A7A7A"),
        plot_bgcolor="rgba(0,0,0,0)",
        paper_bgcolor="rgba(0,0,0,0)",
        margin=dict(l=90, r=70, t=40, b=110),
        annotations=[
            dict(
                text=f"Time: {current_date}",
                x=0.98,
                y=-0.17,
                xref="paper",
                yref="paper",
                showarrow=False,
                xanchor="right",
                font=dict(size=28, color="#7A7A7A", family="Times New Roman")
            )
        ]
    )

    fig_frame.update_xaxes(
        title="Expected market return",
        range=[x_min - x_pad, x_max + x_pad],
        showgrid=False,
        zeroline=False,
        showline=True,
        linecolor="#BFBFBF",
        tickfont=dict(color="#7A7A7A", size=16),
        title_font=dict(color="#7A7A7A", size=22)
    )

    fig_frame.update_yaxes(
        title="CSAD",
        range=[0, y_max + y_pad],
        showgrid=False,
        zeroline=False,
        showline=True,
        linecolor="#BFBFBF",
        tickfont=dict(color="#7A7A7A", size=16),
        title_font=dict(color="#7A7A7A", size=22)
    )

    fig_frame.write_image(
        str(FRAME_DIR / f"frame_{out_i:04d}.png"),
        scale=2
    )

print("Frames saved to:", FRAME_DIR)

# Convert PNG frames to transparent GIF
frame_files = sorted(FRAME_DIR.glob("frame_*.png"))

if not frame_files:
    raise RuntimeError("No frames were created. GIF cannot be generated.")

frames = []

for k in range(1, len(anime_df) + 1):
    current_date = anime_df.index[k - 1]
    current_date_str = current_date.strftime("%Y-%m-%d")

    # Only show event stars whose dates have already happened
    shown_events = event_df[event_df["date"] <= current_date]

    frames.append(
        go.Frame(
            data=[
                # Trace 0: animated CSAD scatter points
                go.Scatter(
                    x=anime_df["CRIX"].iloc[:k],
                    y=anime_df["csad"].iloc[:k],
                    mode="markers",
                    marker=dict(
                        size=10,
                        color=colors[:k],
                    ),
                    hovertemplate=
                        "Expected market return: %{x:.4f}<br>" +
                        "CSAD: %{y:.4f}<extra></extra>"
                ),

                # Trace 1: event stars appear only after event date
                go.Scatter(
                    x=shown_events["x"],
                    y=shown_events["y"],
                    mode="markers+text",
                    text=shown_events["label"],
                    textposition="top center",
                    marker=dict(
                        symbol="star",
                        size=22,
                        color=EVENT_COLOR_PLOTLY,
                        line=dict(color=EVENT_COLOR_PLOTLY, width=1),
                    ),
                    textfont=dict(
                        family="Times New Roman",
                        size=13,
                        color="rgb(50,70,110)"
                    ),
                    hoverinfo="skip"
                )
            ],
            traces=[0, 1],  # animate both main dots and event stars
            name=str(k),
            layout=go.Layout(
                annotations=[
                    dict(
                        text=f"Time:{current_date_str}",
                        x=0.95,
                        y=-0.15,
                        xref="paper",
                        yref="paper",
                        showarrow=False,
                        font=dict(
                            family="Times New Roman",
                            size=24,
                            color="rgb(50,70,110)"
                        )
                    )
                ]
            )
        )
    )

fig.frames = frames

In [ ]:
from pathlib import Path
from PIL import Image

BASE = Path("/Users/HUBerlin/Desktop/Herding/Herding-main 2")

# Use the updated frame folder you already created
FRAME_DIR = BASE / "csad_scatter_gif_frames_UPDATED"

# Save as a NEW updated GIF file
GIF_PATH = BASE / "csad_scatter_transparent_UPDATED.gif"

frame_files = sorted(FRAME_DIR.glob("frame_*.png"))

print("Frame folder:", FRAME_DIR)
print("Number of frames:", len(frame_files))

if not frame_files:
    raise RuntimeError(f"No frames found in {FRAME_DIR}")

# Target around 20 seconds
target_duration_sec = 20
duration_ms = int((target_duration_sec / len(frame_files)) * 1000)

gif_frames = []

for f in frame_files:
    img = Image.open(f).convert("RGBA")

    # Preserve transparency for GIF
    alpha = img.getchannel("A")
    paletted = img.convert("P", palette=Image.ADAPTIVE, colors=255)

    transparency_mask = Image.eval(alpha, lambda a: 255 if a <= 5 else 0)
    paletted.paste(255, mask=transparency_mask)
    paletted.info["transparency"] = 255

    gif_frames.append(paletted)

gif_frames[0].save(
    GIF_PATH,
    save_all=True,
    append_images=gif_frames[1:],
    duration=duration_ms,
    loop=0,
    transparency=255,
    disposal=2,
    optimize=False
)

print("Saved updated GIF:")
print(GIF_PATH)
print("Exists:", GIF_PATH.exists())
print("Size MB:", round(GIF_PATH.stat().st_size / 1024 / 1024, 2))
print("Approx duration seconds:", round(len(frame_files) * duration_ms / 1000, 1))

In [ ]:
import statsmodels.api as sm

Y = csad_df['csad']
X = csad_df[['abs_CRIX', 'sq_CRIX']]

# Add a constant to the independent variables (X) for the intercept
X = sm.add_constant(X)

# Fit the OLS model
model = sm.OLS(Y, X).fit()

# Print the summary of the regression
print(model.summary())